<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/14_learning_to_write_like_shakespeare_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 14: Learning to Write Like Shakespeare - Long Short-Term Memory**

This notebook summarizes Chapter 14 of *Grokking Deep Learning*:

1. Character Language Modeling
2. The Need for Truncated Backpropagation
3. Truncated Backpropagation
4. A Sample of the Output
5. Vanishing and Exploding Gradients
6. A Toy Example of RNN Backpropagation
7. Long Short-Term Memory (LSTM) Cells
8. Some Intuition about LSTM Gates
9. The Long Short-Term Memory Layer
10. Upgrading the Character Language Model
11. Training the LSTM Character Language Model
12. Tuning the LSTM Character Language Model
13. Summary

At the end of Chapters 12 and 13, vanilla recurrent neural networks (RNNs) were trained on a simple series-prediction problem built from synthetic, rule-generated phrases. This chapter moves to a much harder dataset, the works of Shakespeare, and changes the task: the model reads **characters** and learns to predict the **next character**. Working at this scale makes **truncated backpropagation** necessary. The chapter then uses a small numerical experiment to show why vanilla RNNs suffer from **vanishing and exploding gradients**, introduces the **long short-term memory (LSTM)** cell as the standard remedy, and finally retrains the character model with an LSTM.

The chapter opens with a line from Shakespeare's *A Midsummer Night's Dream*: "Lord, what fools these mortals be!"

### **What the code listings of this chapter assume**

- **The framework of the preceding chapter.** The listings use the classes built there (`Tensor` with autograd, `Layer`, `Linear`, `Embedding`, `RNNCell`, `CrossEntropyLoss`, and `SGD`). The chapter does not print them again. The setup cell before the first code cell, which is **not part of the book's code**, defines them by copying their final versions from the Chapter 13 notebook, so that this notebook can run on its own.
- **A text file named `shakespear.txt`** (spelled this way in the book) in the working directory, containing the works of Shakespeare. The book credits `http://karpathy.github.io/2015/05/21/rnn-effectiveness/` as the source of this text. The setup cell before the first code cell, which is **not part of the book's code**, downloads the file from the book's public repository.
- **Running time.** The training cells are long-running: the author reports that the final LSTM model trained overnight (about 8 hours).
- Where a printed listing could not be run as it appears in the book, the smallest possible repair was made, and each repair is documented in a note next to its cell.

---
## **1. Character Language Modeling**

**Let's tackle a more challenging task with the RNN.**

In Chapters 12 and 13 the vanilla RNNs learned a simple series-prediction problem, but the data was a toy collection of phrases generated by rules. Here the dataset is far more demanding: the works of Shakespeare. In addition, the model no longer predicts the next *word* from the previous words (as in the preceding chapter). It is trained on **characters**: given the characters observed so far, it must predict the next character.

### **1.1 Preparing the Data**

The first listing reads the text and builds the vocabulary. In Chapters 12 and 13 the vocabulary consisted of the words of the dataset; now it consists of the **characters** of the dataset. The whole text is accordingly converted into an array of indices, one per character. (The names `word2index` and `word` are kept from the earlier word-level code, although they now refer to characters.) The listing also fixes the random seed, and the book's annotation next to `raw = f.read()` names the source of the text (see the note above).

**Setup (not from the book).** The next cell downloads the text file used by the code that follows.

In [1]:
# Setup (not from the book): download the text file
!wget -q -nc https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/shakespear.txt

**Setup (not from the book).** The next cell defines the framework classes that this chapter uses but does not print. It is the final version of the framework from the book's public repository (`iamtrask/Grokking-Deep-Learning`), which is the version that the code of Chapter 14 was written against. It contains `Tensor`, `SGD`, `Layer`, `Linear`, `Embedding`, `Tanh`, `Sigmoid`, `CrossEntropyLoss`, and `RNNCell`, plus `MSELoss` copied from the Chapter 13 notebook. Compared with the Chapter 13 listings, two things are extended: `Linear` accepts a `bias` option, and `Tensor` has a `softmax` method; the book's code calls both without printing them.

In [2]:
# Setup (not from the book): framework classes of the book's repository
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None

        if(id is None):
            self.id = np.random.randint(0,1000000000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    return
                    print(self.id)
                    print(self.creation_op)
                    print(len(self.creators))
                    for c in self.creators:
                        print(c.creation_op)
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

                if(self.creation_op == "cross_entropy"):
                    dx = self.softmax_output - self.target_dist
                    self.creators[0].backward(Tensor(dx))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def softmax(self):
        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)
        return softmax_output

    def cross_entropy(self, target_indices):

        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)

        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t),-1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * (target_dist)).sum(1).mean()

        if(self.autograd):
            out = Tensor(loss,
                         autograd=True,
                         creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out

        return Tensor(loss)


    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

class Layer(object):

    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters

class SGD(object):

    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):

        for p in self.parameters:

            p.data -= p.grad.data * self.alpha

            if(zero):
                p.grad.data *= 0

class Linear(Layer):

    def __init__(self, n_inputs, n_outputs, bias=True):
        super().__init__()

        self.use_bias = bias

        W = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0/(n_inputs))
        self.weight = Tensor(W, autograd=True)
        if(self.use_bias):
            self.bias = Tensor(np.zeros(n_outputs), autograd=True)

        self.parameters.append(self.weight)

        if(self.use_bias):
            self.parameters.append(self.bias)

    def forward(self, input):
        if(self.use_bias):
            return input.mm(self.weight)+self.bias.expand(0,len(input.data))
        return input.mm(self.weight)

class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # this random initialiation style is just a convention from word2vec
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)

        self.parameters.append(self.weight)

    def forward(self, input):
        return self.weight.index_select(input)

class MSELoss(Layer):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target)*(pred - target)).sum(0)

class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()

class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()

class CrossEntropyLoss(object):

    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        return input.cross_entropy(target)

class RNNCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output, activation='sigmoid'):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        if(activation == 'sigmoid'):
            self.activation = Sigmoid()
        elif(activation == 'tanh'):
            self.activation == Tanh()
        else:
            raise Exception("Non-linearity not found")

        self.w_ih = Linear(n_inputs, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_output)

        self.parameters += self.w_ih.get_parameters()
        self.parameters += self.w_hh.get_parameters()
        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):
        from_prev_hidden = self.w_hh.forward(hidden)
        combined = self.w_ih.forward(input) + from_prev_hidden
        new_hidden = self.activation.forward(combined)
        output = self.w_ho.forward(new_hidden)
        return output, new_hidden

    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)

In [3]:
import sys,random,math
from collections import Counter
import numpy as np
import sys

np.random.seed(0)

f = open('shakespear.txt','r')
raw = f.read()
f.close()

vocab = list(set(raw))
word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i
indices = np.array(list(map(lambda x:word2index[x], raw)))

The last line produces `indices`, a NumPy array that holds the entire dataset as character indices.

### **1.2 The Model and the Optimizer**

The next listing should look familiar. It creates an embedding layer, a vanilla RNN cell whose output size is the number of distinct characters, the cross-entropy loss, and stochastic gradient descent over the parameters of both the model and the embedding layer, with `alpha=0.05`.

*Note:* the accompanying text speaks of embeddings of dimensionality 8 and of output weights initialized to 0, but the printed code uses `dim=512` for the embeddings (the hidden size is 512 as well) and contains no zero-initialization line. The notebook follows the printed code. The zero-initialization appears only later, in the LSTM version of the setup (`model.w_ho.weight.data *= 0`), where the book remarks that it seemed to help training.

In [4]:
embed = Embedding(vocab_size=len(vocab),dim=512)
model = RNNCell(n_inputs=512, n_hidden=512, n_output=len(vocab))

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(),
            alpha=0.05)

---
## **2. The Need for Truncated Backpropagation**

**Backpropagating through 100,000 characters is intractable.**

One of the harder aspects of reading RNN code is the mini-batching logic that feeds in the data. To see what changes, recall the inner loop of the simpler network from the previous chapter. The book reprints it with the inner `for t in range(5)` loop highlighted:

```python
for iter in range(1000):
    batch_size = 100
    total_loss = 0

    hidden = model.init_hidden(batch_size=batch_size)

    for t in range(5):
        input = Tensor(data[0:batch_size,t], autograd=True)
        rnn_input = embed.forward(input=input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)

    target = Tensor(data[0:batch_size,t+1], autograd=True)
    loss = criterion.forward(output, target)
    loss.backward()
    optim.step()
    total_loss += loss.data
    if(iter % 200 == 0):
        p_correct = (target.data == np.argmax(output.data,axis=1)).mean()
        print_loss = total_loss / (len(data)/batch_size)
        print("Loss:",print_loss,"% Correct:",p_correct)
```

*Note:* this listing belongs to the previous chapter and is shown only for comparison, so it is given as a plain block and not as a code cell: it depends on that chapter's toy dataset `data`, which is not part of the setup of this chapter.

Why does the inner loop iterate to 5? The earlier dataset contained no example longer than six words: the network read five words and tried to predict the sixth. The backpropagation step matters even more. When a simple feedforward network classified MNIST digits, the gradients always flowed back through the entire network until they reached the input data, which allowed every weight to adjust in order to predict correctly from the whole input example.

The recurrent case is no different. The network forward-propagates through five inputs, and when `loss.backward()` is called later, the gradients travel all the way back to the input datapoints. This is feasible only because a small number of inputs is fed in at a time. The Shakespeare dataset, however, contains about 100,000 characters, which is far too many to backpropagate through for every single prediction.

The solution is to **not** do it. The network backpropagates through a **fixed number of steps into the past** and then stops. This is called **truncated backpropagation**, and it is the industry standard. The number of steps becomes one more tunable parameter, like the batch size or `alpha`.

---
## **3. Truncated Backpropagation**

**Technically, it weakens the theoretical maximum of the neural network.**

### **3.1 The First Cost: A Shorter Memory**

Truncation shortens the distance over which a network can learn to remember things. If the gradients are cut off after, say, five timesteps, the network cannot learn to remember events that happened more than five timesteps earlier.

The book adds a nuance. Residual information from more than five timesteps back may accidentally remain in the hidden layer of the RNN, but the network cannot use gradients to *ask* the model to keep some specific piece of information from six timesteps ago in order to help the current prediction. In practice, then, networks do not learn to base predictions on input signals older than the truncation length. For language modeling the truncation variable is called `bptt`, and it is usually set somewhere between **16 and 64**. The following listing sets `bptt = 16` together with a batch size of 32 and computes `n_batches`, the length of the stretch of text that each of the 32 columns of the batch will hold:

In [5]:
batch_size = 32
bptt = 16
n_batches = int((indices.shape[0] / (batch_size)))

### **3.2 The Second Cost: More Complicated Mini-Batching**

The other downside is that truncation makes the mini-batching logic more complex. The idea is to pretend that, instead of one big dataset, there is a collection of small datasets of length `bptt`, and the data has to be grouped accordingly:

In [6]:
trimmed_indices = indices[:n_batches*batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches)
batched_indices = batched_indices.transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int(((n_batches-1) / bptt))
input_batches = input_batched_indices[:n_bptt*bptt]
input_batches = input_batches.reshape(n_bptt,bptt,batch_size)
target_batches = target_batched_indices[:n_bptt*bptt]
target_batches =  target_batches.reshape(n_bptt, bptt, batch_size)

There is a lot going on here:

- `trimmed_indices` makes the dataset an even multiple of `batch_size` and `n_batches`, so that when it is grouped into tensors the result is square. (Alternatively, the dataset could be padded with 0s.)
- The next two lines reshape the data so that **each column is a consecutive section of the original `indices` array**.
- `input_batched_indices` drops the last row and `target_batched_indices` drops the first row, so the **targets are the inputs shifted by one row**: at each position the network predicts the next character.
- `n_bptt` is the number of truncated chunks of length `bptt` that fit in the data. The inputs and the targets are cut to that size and reshaped into three-dimensional arrays of shape `(n_bptt, bptt, batch_size)`.

### **3.3 Looking at the Batches**

To make this concrete, the book shows the results as if `batch_size` had been set to **8**, for readability (the real setting is 32). The first cell prints the first five characters of the dataset and their indices:

In [7]:
print(raw[0:5])
print(indices[0:5])

That,
[20 24 52 21  1]


The output printed in the book is:

```
'That,'
array([ 9, 14,  2, 10, 57])
```

These are the first five characters of the dataset, and they spell the string "That,". The next cell prints the first five rows of `batched_indices`:

In [8]:
print(batched_indices[0:5])

[[20 14 11  4 27 24 20  6  6 52 52 40 27  6  1 50  6  8 24 11 40 52  6  6
   6  7  6  7  7  6  6 58]
 [24 50 50  4 61 40 52 22  8 58 18 40 18 21  6 50 36 27 27 50 18  3 36 39
  24 58 54 41 58 41 60  6]
 [52 50 31 40 40 52 18 35 27  6 18 11 39 24  5 49 40 22 18 50 39 40 23 40
  40 57 27  1  6 27 27 36]
 [21 55 20 23  6 23 36  6 22 58  6  6 14 40 40 13 40  1  8 28 35  6 27 40
  23 18 22 50 44 23 41 27]
 [ 1 13  7 31  5  6 27 23  6 27  5 46 50  6  6 38 58  6  6 27 14  8  5 61
   1 52 18 13 22 40 40  8]]


With `batch_size = 8` the book's output is:

```
array([[ 9, 43, 21, 10, 10, 23, 57, 46],
       [14, 44, 39, 21, 43, 14,  1, 10],
       [ 2, 41, 39, 54, 37, 21, 26, 57],
       [10, 39, 57, 48, 21, 54, 38, 43],
       [57, 39, 43,  1, 10, 21, 21, 33]])
```

The book highlights the first column in bold: the indices of the phrase "That," (9, 14, 2, 10, 57) run down the first column on the left. This is a standard construction. There are eight columns because the batch size is 8. This tensor is then used to build a list of smaller datasets, each of length `bptt`.

The next cell shows how the inputs and targets are formed. The target indices are the input indices **offset by one row**, so that the network learns to predict the next character:

In [9]:
print(input_batches[0][0:5])

print(target_batches[0][0:5])

[[20 14 11  4 27 24 20  6  6 52 52 40 27  6  1 50  6  8 24 11 40 52  6  6
   6  7  6  7  7  6  6 58]
 [24 50 50  4 61 40 52 22  8 58 18 40 18 21  6 50 36 27 27 50 18  3 36 39
  24 58 54 41 58 41 60  6]
 [52 50 31 40 40 52 18 35 27  6 18 11 39 24  5 49 40 22 18 50 39 40 23 40
  40 57 27  1  6 27 27 36]
 [21 55 20 23  6 23 36  6 22 58  6  6 14 40 40 13 40  1  8 28 35  6 27 40
  23 18 22 50 44 23 41 27]
 [ 1 13  7 31  5  6 27 23  6 27  5 46 50  6  6 38 58  6  6 27 14  8  5 61
   1 52 18 13 22 40 40  8]]
[[24 50 50  4 61 40 52 22  8 58 18 40 18 21  6 50 36 27 27 50 18  3 36 39
  24 58 54 41 58 41 60  6]
 [52 50 31 40 40 52 18 35 27  6 18 11 39 24  5 49 40 22 18 50 39 40 23 40
  40 57 27  1  6 27 27 36]
 [21 55 20 23  6 23 36  6 22 58  6  6 14 40 40 13 40  1  8 28 35  6 27 40
  23 18 22 50 44 23 41 27]
 [ 1 13  7 31  5  6 27 23  6 27  5 46 50  6  6 38 58  6  6 27 14  8  5 61
   1 52 18 13 22 40 40  8]
 [ 6 43 35 39  7 40 21 40  5  6 40 24 50 39  5 26  6 41 36 35 50 27  6  6
   6  5 39 58  4

The book's output (again with `batch_size = 8`) is:

```
array([[ 9, 43, 21, 10, 10, 23, 57, 46],
       [14, 44, 39, 21, 43, 14,  1, 10],
       [ 2, 41, 39, 54, 37, 21, 26, 57],
       [10, 39, 57, 48, 21, 54, 38, 43],
       [57, 39, 43,  1, 10, 21, 21, 33]])

array([[14, 44, 39, 21, 43, 14,  1, 10],
       [ 2, 41, 39, 54, 37, 21, 26, 57],
       [10, 39, 57, 48, 21, 54, 38, 43],
       [57, 39, 43,  1, 10, 21, 21, 33],
       [43, 43, 41, 60, 52, 12, 54,  1]])
```

The first array is the input and the second is the target; the second starts one row further down in the data. The book reassures the reader that it is fine if this does not fully make sense yet: it has little to do with deep learning theory and is simply a rather complicated part of setting up RNNs that one meets from time to time.

*Notes:* the cells above run with the notebook's `batch_size = 32`, so their output has 32 columns and not the 8 shown in the book. The exact integers also depend on how `list(set(raw))` orders the characters, so they need not match the ones printed in the book.

### **3.4 Iterating with Truncated Backpropagation**

The next listing shows truncated backpropagation in practice. It resembles the iteration logic of Chapter 13. The real differences are that a `batch_loss` is produced at every timestep, and that after every `bptt` steps the network backpropagates and updates its weights. Then the program keeps reading through the dataset as if nothing had happened, **even reusing the hidden state from before**, which is reset only at the start of each epoch (each iteration of the outer loop).

Other details of the listing:

- At the start of each chunk the hidden state is wrapped in a fresh `Tensor` that carries over its values.
- The per-step losses are collected in `losses`, and a running sum of them is kept in `loss`.
- The logging line shows the iteration, the batch number out of the total (195 in the book's output), and the loss `np.exp(total_loss / (batch_i+1))`. At the first batch of each epoch it appends a 70-character sample produced by `generate_sample`, which is defined in the next section. The line is written to the screen only for every tenth batch.
- After every epoch the learning rate is multiplied by 0.99.

Because the function calls `generate_sample`, that function must exist before `train` is called; the call itself is therefore placed in the next section, after `generate_sample` is defined.

*Notes on this cell:* (1) in the book's typesetting the line `target = Tensor(target_batches[batch_i][t], autograd=True)` is indented one character less than its neighbours, which Python rejects (inconsistent indentation); the notebook indents it like the lines around it. (2) The book prints the call `train()` directly below the function; it is moved to the next section, as just explained. (3) The loop `for loss in losses:` with the body `""` does nothing else than rebind `loss` to the last element of `losses`; it is kept exactly as printed.

In [10]:
def train(iterations=100):
    for iter in range(iterations):
        total_loss = 0
        n_loss = 0

        hidden = model.init_hidden(batch_size=batch_size)
        for batch_i in range(len(input_batches)):

            hidden = Tensor(hidden.data, autograd=True)
            loss = None
            losses = list()
            for t in range(bptt):
                input = Tensor(input_batches[batch_i][t], autograd=True)
                rnn_input = embed.forward(input=input)
                output, hidden = model.forward(input=rnn_input,
                                               hidden=hidden)
                target = Tensor(target_batches[batch_i][t], autograd=True)
                batch_loss = criterion.forward(output, target)
                losses.append(batch_loss)
                if(t == 0):
                    loss = batch_loss
                else:
                    loss = loss + batch_loss
            for loss in losses:
                ""
            loss.backward()
            optim.step()
            total_loss += loss.data
            log = "\r Iter:" + str(iter)
            log += " - Batch "+str(batch_i+1)+"/"+str(len(input_batches))
            log += " - Loss:" + str(np.exp(total_loss / (batch_i+1)))
            if(batch_i == 0):
                log += " - " + generate_sample(70,'\n').replace("\n"," ")
            if(batch_i % 10 == 0 or batch_i-1 == len(input_batches)):
                sys.stdout.write(log)
        optim.alpha *= 0.99
        print()

---
## **4. A Sample of the Output**

**By sampling from the predictions of the model, you can write Shakespeare!**

The next listing reuses a subset of the training logic to make predictions with the model. The predictions are collected in a string, which the function returns. The generated sample looks quite Shakespearean and even includes characters talking.

The function works as follows:

1. It starts from a given character (`init_char`) and a freshly initialized hidden state for a batch of one.
2. At every step it embeds the current character, runs the model, and obtains the scores for the next character.
3. The scores are multiplied by 10 before the softmax. The book's annotation calls this the **temperature for sampling**: a higher value makes the choice greedier.
4. The line `m = (temp_dist > np.random.rand()).argmax()` is annotated as **sampling from the predictions** (`pred`), and gives the index `m` of the next character.
5. That character is appended to the output string and also becomes the next input.

In [11]:
def generate_sample(n=30, init_char=' '):
    s = ""
    hidden = model.init_hidden(batch_size=1)
    input = Tensor(np.array([word2index[init_char]]))
    for i in range(n):
        rnn_input = embed.forward(input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)
        output.data *= 10
        temp_dist = output.softmax()
        temp_dist /= temp_dist.sum()

        m = (temp_dist > np.random.rand()).argmax()
        c = vocab[m]
        input = Tensor(np.array([m]))
        s += c
    return s

Now that `generate_sample` exists, `train()` can be called. This is the call that the book prints right below the definition of `train`.

The training log printed in the book shows the loss falling as training proceeds: about 148.0 in Iter 0 (batch 191 of 195), about 20.6 in Iter 1 (batch 191 of 195), and about 1.05 in Iter 99 (batch 61 of 195). The 70-character sample printed at the start of each epoch also improves, from nonsense such as `mhnethet tttttt t t t` in Iter 1 to the more word-like `I af the mands your` in Iter 99.

*Note:* with 100 epochs this cell takes a long time to run; the number of epochs is the default argument `iterations=100` of `train`.

In [12]:
train()

 Iter:0 - Batch 191/195 - Loss:50.571967129918974
 Iter:1 - Batch 191/195 - Loss:17.206954819930885
 Iter:2 - Batch 191/195 - Loss:13.801869167442513
 Iter:3 - Batch 191/195 - Loss:12.08009391485899
 Iter:4 - Batch 191/195 - Loss:11.01536223499241
 Iter:5 - Batch 101/195 - Loss:10.39629951309195

KeyboardInterrupt: 

After training, the book generates a longer sample of 2,000 characters, starting from a newline:

In [13]:
print(generate_sample(n=2000, init_char='\n'))

ot wot wot wot wve wove vove vouve voon the wove wove wow the woulve the wous vove wove voon the wous vove voull wove wouve voon the wouve woull woull wous wous wous wveand woull wove wous wove vow the wove wouve vow vove wous wove vouve wous woull wous wove vove woull wous wove woull wow the wove vove wove wouve wow wove wove vove vove wove wouve wove vow the wous with wous wveve wous wveand wove vove wove wove wove voo wot wove wous vouve wveve woull wow the woull wous vove wow the wove wveve wouve wous wouve wous wove vow wot wove wveve vove vove voove vow the vove woull wveve with vove woull woull wous wove vove wove wove woulve the wow the with wove with wow the wove vove wowve the vouve vouve voon the voon the wouve vow the wove voon the wove wouve wove wow the wous wove wove wous wous woulve vove vouve wous with wouve vove wove wow the wove woull wove vow wot wot wot wot with vove wve wove wive wouve wveve wove wous voon the voull wove wous wous wove wove wous wove wove wove wow

The book prints an excerpt of the result. It is not real English, but its structure is strikingly like that of a play: speaker names in capitals followed by a colon (for example `CHENRO:`, `ANG, makes:`, `DESS:`), short verse-like lines, and invented words that nevertheless look plausible (for example `Plirt`, `confectife`, `nightion`). Some phrases are even repeated, such as "my goth Plish love". The reader's own output will differ, because the sampling is random and the training run is different.

---
## **5. Vanishing and Exploding Gradients**

**Vanilla RNNs suffer from vanishing and exploding gradients.**

The book recalls the images from the first time an RNN was assembled. The goal was to combine word embeddings **in a way that order matters**. This was achieved by learning a matrix that transforms the running vector from one timestep to the next.

> The embeddings of "Red Sox defeat Yankees" are stacked and added together, step by step, into a single vector for the whole phrase
>
> <img src="pictures/chapter 14/picture 14.1.png" alt="Figure 14.1" width="600"/>

> The same phrase with a weight matrix (x) applied between steps: the running vector is multiplied by the matrix and the next word embedding is then added (+)
>
> <img src="pictures/chapter 14/picture 14.2.png" alt="Figure 14.2" width="600"/>

Forward propagation therefore became a **two-step process**: take the embedding of the first word ("Red" in the example), multiply it by the weight matrix, and add the embedding of the next word ("Sox"). The resulting vector is multiplied by the **same** weight matrix, the next word is added, and so on until the whole series of words has been read.

As already known, a **nonlinearity** is added when the hidden state is generated, so forward propagation is really a **three-step process**: multiply the previous hidden state by a weight matrix, add the embedding of the next word, and apply a nonlinearity.

This nonlinearity plays an important role in the **stability** of the network. However long the sequence of words is, the hidden states (which could in theory grow larger and larger over time) are forced to stay within the range of the nonlinearity, between 0 and 1 in the case of a sigmoid.

Backpropagation, however, proceeds in a slightly different way, and it does **not** have this nice property. It tends to produce either extremely large or extremely small values. Large values can cause **divergence** (lots of not-a-number values, NaNs), whereas extremely small values keep the network from learning. The next section looks at RNN backpropagation more closely.

---
## **6. A Toy Example of RNN Backpropagation**

**To see vanishing and exploding gradients firsthand, let's synthesize an example.**

The following listing runs a recurrent backpropagation loop for sigmoid and relu activations. Notice how the gradients become very small with the sigmoid and very large with the relu. During backpropagation they grow because of the **matrix multiplication**, and they shrink because the **sigmoid has a very flat derivative at its tails**, which is common for many nonlinearities. The book's annotations on the code say the same thing:

- The derivative of the sigmoid causes very small gradients when the activation is very near 0 or 1 (the tails).
- The matrix multiplication causes exploding gradients that are not squished by a nonlinearity (as with the sigmoid).

The listing first applies the same weight matrix `[[1,4],[4,1]]` ten times in a forward loop with a sigmoid, then backpropagates through those ten steps. It then repeats the experiment with a relu.

In [14]:
(sigmoid,relu)=(lambda x:1/(1+np.exp(-x)), lambda x:(x>0).astype(float)*x)
weights = np.array([[1,4],[4,1]])
activation = sigmoid(np.array([1,0.01]))

print("Sigmoid Activations")
activations = list()
for iter in range(10):
    activation = sigmoid(activation.dot(weights))
    activations.append(activation)
    print(activation)
print("\nSigmoid Gradients")
gradient = np.ones_like(activation)
for activation in reversed(activations):
    gradient = (activation * (1 - activation) * gradient)
    gradient = gradient.dot(weights.transpose())
    print(gradient)

print("Activations")
activations = list()
for iter in range(10):
    activation = relu(activation.dot(weights))
    activations.append(activation)
    print(activation)
print("\nGradients")
gradient = np.ones_like(activation)
for activation in reversed(activations):
    gradient = ((activation > 0) * gradient).dot(weights.transpose())
    print(gradient)

Sigmoid Activations
[0.93940638 0.96852968]
[0.9919462  0.99121735]
[0.99301385 0.99302901]
[0.9930713  0.99307098]
[0.99307285 0.99307285]
[0.99307291 0.99307291]
[0.99307291 0.99307291]
[0.99307291 0.99307291]
[0.99307291 0.99307291]
[0.99307291 0.99307291]

Sigmoid Gradients
[0.03439552 0.03439552]
[0.00118305 0.00118305]
[4.06916726e-05 4.06916726e-05]
[1.39961115e-06 1.39961115e-06]
[4.81403643e-08 4.81403637e-08]
[1.65582672e-09 1.65582765e-09]
[5.69682675e-11 5.69667160e-11]
[1.97259346e-12 1.97517920e-12]
[8.45387597e-14 8.02306381e-14]
[1.45938177e-14 2.16938983e-14]
Activations
[4.8135251  4.72615519]
[23.71814585 23.98025559]
[119.63916823 118.852839  ]
[595.05052421 597.40951192]
[2984.68857188 2977.61160877]
[14895.13500696 14916.36589628]
[74560.59859209 74496.90592414]
[372548.22228863 372739.30029248]
[1863505.42345854 1862932.18944699]
[9315234.18124649 9316953.88328115]

Gradients
[5. 5.]
[25. 25.]
[125. 125.]
[625. 625.]
[3125. 3125.]
[15625. 15625.]
[78125. 78125.]


### **6.1 Reading the Output**

- **Sigmoid.** The activations saturate quickly: they go from about 0.939 and 0.969 in the first step to about 0.9919 and 0.9912, and then settle at 0.99307291 for both units. The gradients, in contrast, **vanish**: about 0.034 after the first backward step, about 0.0012 after the second, about 4.07e-05 after the third, and only about 1.46e-14 and 2.17e-14 after the tenth.
- **Relu.** The activations grow by a large factor at every step, and the gradients **explode**: they equal 5, 25, 125, and so on, up to 9,765,625 after ten steps.

*Note:* the book's printed table of relu activations starts at `[23.71814585 23.98025559]` and ends at about 46.6 million. When the listing is run exactly as printed, the first relu activation is `[4.8135251 4.72615519]` (the second one is the `[23.718... 23.980...]` shown in the book) and the tenth is about 9.3 million. The qualitative message (explosion) is the same, and the printed gradients and the sigmoid values agree with the book.

---
## **7. Long Short-Term Memory (LSTM) Cells**

**LSTMs are the industry standard model to counter vanishing and exploding gradients.**

The previous section showed that vanishing and exploding gradients come from the way the hidden state of an RNN is updated: the **combination of a matrix multiplication and a nonlinearity** is used to form the next hidden state. The solution offered by LSTMs is surprisingly simple.

> **The gated copy trick.** An LSTM creates the next hidden state by **copying** the previous hidden state and then **adding or removing information** as necessary. The mechanisms it uses to add and remove information are called **gates**.

### **7.1 From the RNN Cell to the LSTM Cell**

For comparison, here is the forward propagation logic of the RNN cell (a method of the cell class, so it is shown here as a plain block): the previous hidden state is multiplied by a weight matrix, the transformed input is added, the activation is applied to obtain the new hidden state, and the output is computed from it.

```python
def forward(self, input, hidden):
    from_prev_hidden = self.w_hh.forward(hidden)
    combined = self.w_ih.forward(input) + from_prev_hidden
    new_hidden = self.activation.forward(combined)
    output = self.w_ho.forward(new_hidden)
    return output, new_hidden
```

The new forward propagation logic of the LSTM cell is shown next (again a method excerpt, whose complete runnable version appears in Section 9). The LSTM has **two hidden state vectors**: `h` (for "hidden") and `cell`. The one that matters is `cell`. In the book the line that updates it is highlighted:

```python
def forward(self, input, hidden):

    prev_hidden, prev_cell = (hidden[0], hidden[1])

    f = (self.xf.forward(input) + self.hf.forward(prev_hidden)).sigmoid()
    i = (self.xi.forward(input) + self.hi.forward(prev_hidden)).sigmoid()
    o = (self.xo.forward(input) + self.ho.forward(prev_hidden)).sigmoid()
    u = (self.xc.forward(input) + self.hc.forward(prev_hidden)).tanh()
    cell = (f * prev_cell) + (i * u)
    h = o * cell.tanh()
    output = self.w_ho.forward(h)
    return output, (h, cell)
```

The new `cell` is the previous cell, weighted by `f`, **plus** the vector `u`, weighted by `i`:

- `f` is the **forget** gate. If it takes the value 0, the new cell erases what it saw before.
- `i` is the **input** gate. If it is 1, the value of `u` is fully added to create the new cell.
- `u` is the **update** vector (computed with a `tanh`), the value that is added into the cell.
- `o` is the **output** gate. It controls how much of the cell's state the output prediction is allowed to see. If `o` is all zeros, the line `self.w_ho.forward(h)` makes a prediction that ignores the cell state entirely.

---
## **8. Some Intuition about LSTM Gates**

**LSTM gates are semantically similar to reading from and writing to memory.**

There are three gates, `f`, `i`, and `o`, and a cell-update vector `u`; they can be read as **forget, input, output, and update**. They work together so that any information stored or manipulated in the cell `c` can be handled **without any matrix multiplication or nonlinearity being applied to `c` itself** at each update. In other words, the code never calls `nonlinearity(c)` or `c.dot(weights)`.

This is what allows the LSTM to carry information across a time series without suffering from vanishing or exploding gradients. Each step is a **copy** (as long as `f` is nonzero) plus an **update** (as long as `i` is nonzero). The hidden value `h` is then a **masked version of the cell** that is used for making the prediction.

Note also that the three gates are all formed in the same way. Each has its own weight matrices, but each conditions on the **input** and on the **previous hidden state**, and passes the sum through a **sigmoid**. The sigmoid is what makes them so useful as gates: it saturates at 0 and at 1.

```python
f = (self.xf.forward(input) + self.hf.forward(prev_hidden)).sigmoid()
i = (self.xi.forward(input) + self.hi.forward(prev_hidden)).sigmoid()
o = (self.xo.forward(input) + self.ho.forward(prev_hidden)).sigmoid()
```

### **8.1 What About `h`?**

A last possible criticism concerns `h`. It is still prone to vanishing and exploding gradients, because it is used essentially like the hidden state of a vanilla RNN. The book answers in two steps:

- Since the vector `h` is always built from vectors squished by `tanh` and sigmoid, **exploding gradients are not really a problem** for it, only vanishing ones.
- Vanishing gradients in `h` are acceptable, because `h` is conditioned on `c`, and `c` can carry the long-range information that vanishing gradients cannot learn to carry.

All long-range information therefore travels through `c`, while `h` is only a **localized interpretation of `c`**, useful for making an output prediction and for building the gate activations of the following timestep. In short, `c` can learn to transport information over long distances, so it does not matter that `h` cannot.

---
## **9. The Long Short-Term Memory Layer**

**You can use the autograd system to implement an LSTM.**

The LSTM is implemented as a layer class of the framework, `LSTMCell(Layer)`:

- The constructor creates four `Linear` layers that map the input to the hidden size (`xf`, `xi`, `xo`, `xc`, with bias), four that map the previous hidden state to the hidden size (`hf`, `hi`, `ho`, `hc`, without bias), and the output layer `w_ho` (without bias). All their parameters are collected in `self.parameters`.
- `forward` receives the pair `(h, cell)` as `hidden`, computes the three gates and the update vector, builds the new cell and the new `h`, and returns the output together with the new pair. In this listing the update vector is called `g` and the cell is called `c`, instead of `u` and `cell` in the explanation of the previous sections.
- `init_hidden` creates `h` and `c` as zero tensors of shape `(batch_size, n_hidden)`, sets the first column of each to 1, and returns them as a tuple.

*Note:* in the book's listing the final line `return (h, c)` is indented less than the lines of `init_hidden`, which breaks the code (indentation error). It obviously belongs to `init_hidden`, so the notebook indents it by 8 spaces to match the body of that method.

In [15]:
class LSTMCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output):
        super().__init__()
        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        self.xf = Linear(n_inputs, n_hidden)
        self.xi = Linear(n_inputs, n_hidden)
        self.xo = Linear(n_inputs, n_hidden)
        self.xc = Linear(n_inputs, n_hidden)
        self.hf = Linear(n_hidden, n_hidden, bias=False)
        self.hi = Linear(n_hidden, n_hidden, bias=False)
        self.ho = Linear(n_hidden, n_hidden, bias=False)
        self.hc = Linear(n_hidden, n_hidden, bias=False)

        self.w_ho = Linear(n_hidden, n_output, bias=False)

        self.parameters += self.xf.get_parameters()
        self.parameters += self.xi.get_parameters()
        self.parameters += self.xo.get_parameters()
        self.parameters += self.xc.get_parameters()
        self.parameters += self.hf.get_parameters()
        self.parameters += self.hi.get_parameters()
        self.parameters += self.ho.get_parameters()
        self.parameters += self.hc.get_parameters()

        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):

        prev_hidden = hidden[0]
        prev_cell = hidden[1]

        f=(self.xf.forward(input)+self.hf.forward(prev_hidden)).sigmoid()
        i=(self.xi.forward(input)+self.hi.forward(prev_hidden)).sigmoid()
        o=(self.xo.forward(input)+self.ho.forward(prev_hidden)).sigmoid()
        g = (self.xc.forward(input) +self.hc.forward(prev_hidden)).tanh()
        c = (f * prev_cell) + (i * g)
        h = o * c.tanh()

        output = self.w_ho.forward(h)
        return output, (h, c)

    def init_hidden(self, batch_size=1):
        h = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        c = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        h.data[:,0] += 1
        c.data[:,0] += 1
        return (h, c)

---
## **10. Upgrading the Character Language Model**

**Let's swap out the vanilla RNN with the new LSTM cell.**

Earlier in the chapter a character language model was trained to predict Shakespeare; now an LSTM-based model is trained for the same task. Thanks to the framework of the preceding chapter, this is easy. (The book notes that the complete code is available on the book's website and in its public repository.)

Here is the new setup code. The edits compared with the vanilla RNN code were printed in bold in the book. Hardly anything changed:

- `model = LSTMCell(n_inputs=512, n_hidden=512, n_output=len(vocab))` replaces the RNN cell.
- `model.w_ho.weight.data *= 0` sets the output weights to 0 (the book's annotation says: "This seemed to help training").
- `batch_size = 16` and `bptt = 25` replace 32 and 16.
- `min_loss = 1000` is added at the end, to track the best loss seen so far.

The rest of the listing (reading the data, building the vocabulary, and grouping the data into `bptt`-sized batches) repeats the earlier code.

In [16]:
import sys,random,math
from collections import Counter
import numpy as np
import sys

np.random.seed(0)

f = open('shakespear.txt','r')
raw = f.read()
f.close()

vocab = list(set(raw))
word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i
indices = np.array(list(map(lambda x:word2index[x], raw)))

embed = Embedding(vocab_size=len(vocab),dim=512)
model = LSTMCell(n_inputs=512, n_hidden=512, n_output=len(vocab))
model.w_ho.weight.data *= 0

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(),
            alpha=0.05)

batch_size = 16
bptt = 25
n_batches = int((indices.shape[0] / (batch_size)))

trimmed_indices = indices[:n_batches*batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches)
batched_indices = batched_indices.transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int(((n_batches-1) / bptt))
input_batches = input_batched_indices[:n_bptt*bptt]
input_batches = input_batches.reshape(n_bptt,bptt,batch_size)
target_batches = target_batched_indices[:n_bptt*bptt]
target_batches = target_batches.reshape(n_bptt, bptt, batch_size)
min_loss = 1000

---
## **11. Training the LSTM Character Language Model**

**The training logic also has not changed much.**

The only real change from the vanilla RNN logic concerns truncated backpropagation, because there are now **two hidden vectors per timestep** instead of one. The fix is minor: the hidden state is now a pair, and both of its parts are re-wrapped in new `Tensor` objects at the start of each chunk (printed in bold in the book). The book has also added a few "bells and whistles" that make training easier:

- `alpha` **decreases slowly over time** (it is multiplied by 0.99 after every epoch).
- There is **more logging**: the current `alpha`, the minimum loss so far, and a 70-character sample (started from the letter `T`) at the first batch of each epoch.
- The losses of the `bptt` steps are accumulated (`losses.append(batch_loss + losses[-1])`), and the last element, `loss = losses[-1]`, is the one that is backpropagated. The logged loss is `np.exp(total_loss / (batch_i+1))`, where `total_loss` adds `loss.data / bptt` at each batch.

The loop calls `generate_sample` from Section 4; the second version of that function is printed after the training listing, in the next section.

*Notes on this cell:* (1) the book prints the loop as `for iter in range(iterations):` without defining `iterations` (in the earlier listing it was an argument of the function `train`). The notebook adds the single line `iterations = 100` before the loop, the default of that earlier function. At the book's scale this run takes hours, so a smaller number may be used for a quick trial. (2) As in the earlier listing, one line (`output, hidden = model.forward(...)`) is indented one character less than its neighbours in the book, which Python rejects; the notebook aligns it with the lines around it.

In [17]:
iterations = 100
for iter in range(iterations):
    total_loss, n_loss = (0, 0)

    hidden = model.init_hidden(batch_size=batch_size)
    batches_to_train = len(input_batches)

    for batch_i in range(batches_to_train):

        hidden = (Tensor(hidden[0].data, autograd=True),
                  Tensor(hidden[1].data, autograd=True))
        losses = list()

        for t in range(bptt):
            input = Tensor(input_batches[batch_i][t], autograd=True)
            rnn_input = embed.forward(input=input)
            output, hidden = model.forward(input=rnn_input, hidden=hidden)

            target = Tensor(target_batches[batch_i][t], autograd=True)
            batch_loss = criterion.forward(output, target)

            if(t == 0):
                losses.append(batch_loss)
            else:
                losses.append(batch_loss + losses[-1])
        loss = losses[-1]

        loss.backward()
        optim.step()

        total_loss += loss.data / bptt
        epoch_loss = np.exp(total_loss / (batch_i+1))
        if(epoch_loss < min_loss):
            min_loss = epoch_loss
            print()
        log = "\r Iter:" + str(iter)
        log += " - Alpha:" + str(optim.alpha)[0:5]
        log += " - Batch "+str(batch_i+1)+"/"+str(len(input_batches))
        log += " - Min Loss:" + str(min_loss)[0:5]
        log += " - Loss:" + str(epoch_loss)
        if(batch_i == 0):
            s = generate_sample(n=70, init_char='T').replace("\n"," ")
            log += " - " + s
        sys.stdout.write(log)
    optim.alpha *= 0.99


 Iter:0 - Alpha:0.05 - Batch 1/249 - Min Loss:62.00 - Loss:62.000000000000064 - vvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvv
 Iter:0 - Alpha:0.05 - Batch 2/249 - Min Loss:61.99 - Loss:61.99905884951485
 Iter:0 - Alpha:0.05 - Batch 3/249 - Min Loss:61.98 - Loss:61.989726345425225
 Iter:0 - Alpha:0.05 - Batch 4/249 - Min Loss:61.97 - Loss:61.97275624062324
 Iter:0 - Alpha:0.05 - Batch 5/249 - Min Loss:61.94 - Loss:61.94163117510397
 Iter:0 - Alpha:0.05 - Batch 6/249 - Min Loss:61.88 - Loss:61.88134258619446
 Iter:0 - Alpha:0.05 - Batch 7/249 - Min Loss:61.77 - Loss:61.7799038953562
 Iter:0 - Alpha:0.05 - Batch 8/249 - Min Loss:61.53 - Loss:61.53739795328808
 Iter:0 - Alpha:0.05 - Batch 9/249 - Min Loss:61.02 - Loss:61.023889196093634
 Iter:0 - Alpha:0.05 - Batch 10/249 - Min Loss:60.27 - Loss:60.27174739051778
 Iter:0 - Alpha:0.05 - Batch 11/249 - Min Loss:58.76 - Loss:58.766535924156706
 Iter:0 - Alpha:0.05 - Batch 12/249 - Min Loss:56.81 - Loss:56.81384682294

KeyboardInterrupt: 

---
## **12. Tuning the LSTM Character Language Model**

**I spent about two days tuning this model, and it trained overnight.**

The author reports that the model took a very long time to train, because it has a lot of parameters. The author also had to train it many times to find a good tuning (learning rate, batch size, and so on) for this task, and the final model trained overnight (8 hours). In general, **the longer you train, the better the results**.

Some of the training output printed in the book (from the author's own tuned run, so the exact values need not coincide with what the listing above prints) is:

```
I:0 - Alpha:0.05 - Batch 1/249 - Min Loss:62.00 - Loss:62.00 - eeeeeeeeee
                                   ...
I:7 - Alpha:0.04 - Batch 140/249 - Min Loss:10.5 - Loss:10.7 - heres, and
                                   ...
I:91 - Alpha:0.016 - Batch 176/249 - Min Loss:9.900 - Loss:11.9757225699
```

At the start (iteration 0) the loss is 62 and the sample is just a string of the letter `e`. By iteration 7 the minimum loss has fallen to about 10.5 and the sample has become word-like (`heres, and`). At iteration 91 the minimum loss is about 9.9, while the current loss is about 12.0.

### **12.1 Generating Text from the Trained LSTM**

The book then redefines `generate_sample` for the LSTM. This version differs from the earlier one in two ways. The scores are multiplied by 15 instead of 10, and the next character is no longer drawn at random: the line `m = output.data.argmax()`, annotated "takes the max prediction", always picks the character with the highest score. (The softmax lines remain in the function, but the choice is made directly from `output.data`.) The listing then prints a sample of 500 characters, starting from a newline.

In [18]:
def generate_sample(n=30, init_char=' '):
    s = ""
    hidden = model.init_hidden(batch_size=1)
    input = Tensor(np.array([word2index[init_char]]))
    for i in range(n):
        rnn_input = embed.forward(input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)
        output.data *= 15
        temp_dist = output.softmax()
        temp_dist /= temp_dist.sum()

        m = output.data.argmax()
        c = vocab[m]
        input = Tensor(np.array([m]))
        s += c
    return s
print(generate_sample(n=500, init_char='\n'))



OLELI

O:

OLO:

O:

O INLe te tout tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont tont to


The sample printed in the book begins with `Intestay thee.` and includes speaker labels such as `SIR:` and `KENNOR:`. It also shows a typical weakness of always taking the most likely character: word fragments repeat, for example "the see" and "the sentastar". Like the first sample, it is not real English, but its layout resembles a play.

---
## **13. Summary**

**LSTMs are incredibly powerful models.**

The distribution of Shakespearean language that the LSTM learned to generate should not be taken lightly. Language is an extremely complex statistical distribution to learn, and the fact that LSTMs do so well still baffles the author (and others). At the time of writing they are the state-of-the-art approach by a wide margin. Small variants of this model either are, or recently have been, the state of the art in a wide variety of tasks and, together with **word embeddings** and **convolutional layers**, will undoubtedly remain one of the go-to tools for a long time.

The key takeaways of Chapter 14 are:

- A **character language model** uses the characters of a text as its vocabulary, converts the whole dataset into character indices, and learns to predict the next character from the previous ones. Here it is trained on the works of Shakespeare (about 100,000 characters).
- Backpropagating through the entire dataset is intractable, so **truncated backpropagation** is used: gradients flow back only a fixed number of steps (`bptt`, typically 16 to 64), and the hidden state is carried over from one chunk to the next. The price is that the network cannot learn dependencies longer than the truncation length, and the mini-batching logic becomes more complicated (inputs and targets are the same data offset by one row, grouped into tensors of shape `(n_bptt, bptt, batch_size)`).
- **Sampling** from the predictions of the model (with a temperature that scales the scores) produces text that looks like a play, including speaker names, although many words are invented.
- Vanilla RNNs suffer from **vanishing and exploding gradients**. The hidden states stay within the range of the nonlinearity during forward propagation, but during backpropagation the gradients become extremely small (sigmoid, whose derivative is flat at its tails) or extremely large (repeated matrix multiplication, as the relu example shows).
- The **LSTM** counters this with the **gated copy trick**: the new cell state is a copy of the previous one, scaled by the **forget** gate `f`, plus an **update** `u` scaled by the **input** gate `i`, so no matrix multiplication or nonlinearity is applied to the cell itself. The **output** gate `o` controls how much of the cell the prediction may see. The long-range information travels in the cell `c`, while `h` is only a localized interpretation of it.
- The three gates are formed in the same way (own weights, conditioned on the input and the previous hidden state, passed through a sigmoid that saturates at 0 and 1), and `h` is exposed mainly to vanishing, not exploding, gradients because it is built from `tanh` and sigmoid outputs.
- With the framework of the preceding chapter, an LSTM is a layer class (`LSTMCell`) whose hidden state is a pair `(h, c)`; swapping it in for the RNN cell needs only small changes to the setup and to the truncated backpropagation logic. Tuning (learning rate, batch size, and so on) took the author about two days, and the final model trained overnight (about 8 hours); in general, the longer the training, the better the results.